In [34]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import os
import glob
from PIL import Image
import numpy as np

# 1. Загрузка и синхронизация данных

In [35]:
possible_paths = [
    "/Users/grushinda/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1",
    #"/kaggle/input/gtzan-dataset-music-genre-classification",
    #"/root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1",
]

dataset_path = None
for path in possible_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    print("Датасет не найден!")
    import sys
    sys.exit(1)

print(f"Датасет найден: {dataset_path}")

# определяем путь к данным
if os.path.exists(os.path.join(dataset_path, "Data")):
    data_path = os.path.join(dataset_path, "Data")
else:
    data_path = dataset_path

image_path = os.path.join(data_path, "images_original")
#csv_path = os.path.join(data_path, "features_30_sec.csv")

print(f"Изображения: {image_path} (существует: {os.path.exists(image_path)})")
#print(f"CSV: {csv_path} (существует: {os.path.exists(csv_path)})")

if not os.path.exists(image_path) : #or not os.path.exists(csv_path):
    print("Ошибка: данные не найдены!")
    import sys
    sys.exit(1)

Датасет найден: /Users/grushinda/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1
Изображения: /Users/grushinda/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data/images_original (существует: True)


In [36]:
image_data = []      # изображения
image_labels = []    # метки из изображений
csv_labels = []      # метки из CSV
valid_filenames = [] # имена файлов для проверки

# проходим по жанрам в папке
class_names = sorted([d for d in os.listdir(image_path) if os.path.isdir(os.path.join(image_path, d))])
print(f"Найдены жанры: {class_names}")

Найдены жанры: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


In [37]:
# Проходим по каждой папке ...
for class_idx, class_name in enumerate(class_names):
    # путь к кккаждой папке ...
    class_dir = os.path.join(image_path, class_name)
    # print(class_dir)
    # путьь к каждому файлу в папке class_dir
    image_files = sorted(glob.glob(os.path.join(class_dir, "*.png")))
    #print(image_files)
    # в цикле по каждому файлу 
    for img_path in image_files:
        # обрезаем расширение 
        img_name = os.path.basename(img_path).replace('.png', '')

        # Загружаем изображения ...
        img = Image.open(img_path).convert('RGB')
        image_data.append(img)
        image_labels.append(class_idx)
        valid_filenames.append(img_name)
print(f"Загруженно: {len(image_data)} изображений")        

Загруженно: 999 изображений


# 2. Трансформация 

In [38]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

def_transforms = weights.transforms()

In [39]:
print(def_transforms.resize_size)

[256]


In [40]:
# трансформации для изображений
# С помощью конвеера выполняем преобразования под требования вх/ данных pytorch
image_transform = transforms.Compose([
    transforms.Resize((224, 224)), # все изображения сжимаем до 224x224 пикселей (размер в батче должен быть одинаковый)
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=2),
    transforms.ToTensor(), # преобразукм в тензор + стандартизация 
    transforms.Normalize(mean=def_transforms.mean, std=def_transforms.std) # нормализация сдвиггаем диапазон на (-1,1) out = (in - mean) / std
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)), # Только приведение к размеру
    transforms.ToTensor(), 
    transforms.Normalize(mean=def_transforms.mean, std=def_transforms.std) 
])

# 3. Разделение данных

In [41]:
# делим индексы (а не данные) на три блока
def split_data(n_samples,        # общий объем = 999
               train_ratio=0.7,  # - обучение 70%
               val_ratio=0.15,   # - валидация обучения 15%
               test_ratio=0.15): # - тестовая выборка  15%
    
    indices = np.random.permutation(n_samples) # перемешиваем индексы 
    train_end = int(train_ratio * n_samples)   # получаем кол-во 
    val_end = train_end + int(val_ratio * n_samples) # получаем кол-во 
    
    return indices[:train_end], indices[train_end:val_end], indices[val_end:] 

n_samples = len(image_data) # = 999

train_idx, val_idx, test_idx = split_data(n_samples)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Train: 699, Val: 149, Test: 151


In [ ]:

# преобразуем изображения в тензоры
#X_images_train = torch.stack([image_transform(img) for img in image_data[train_idx]]) 
#y_images_train = torch.tensor(image_labels[train_idx])
X_images_train = torch.stack([image_transform(image_data[int(i)]) for i in train_idx])
y_images_train = torch.tensor([image_labels[int(i)] for i in train_idx])
print(f"  Изображения train: {X_images_train.shape}")

X_images_val = torch.stack([val_transform(image_data[int(i)]) for i in val_idx])
y_images_val = torch.tensor([image_labels[int(i)] for i in val_idx])
print(f"  Изображения val: {X_images_val.shape}")

X_images_test = torch.stack([val_transform(image_data[int(i)]) for i in test_idx])
y_images_test = torch.tensor([image_labels[int(i)] for i in test_idx])
print(f"  Изображения test: {X_images_test.shape}")


# создаём датасеты для изображений
train_images = TensorDataset(X_images_train, y_images_train)
val_images = TensorDataset(X_images_val, y_images_val)
test_images = TensorDataset(X_images_test, y_images_test)

  Изображения train: torch.Size([699, 3, 224, 224])
  Изображения val: torch.Size([149, 3, 224, 224])
  Изображения test: torch.Size([151, 3, 224, 224])


: 

In [ ]:
# DataLoader
batch_size = 64
train_loader_images = DataLoader(train_images, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader_images = DataLoader(val_images, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader_images = DataLoader(test_images, batch_size=batch_size, shuffle=False, num_workers=2)
print(f"Image loaders: Train={len(train_loader_images)}, Val={len(val_loader_images)}, Test={len(test_loader_images)}")

Image loaders: Train=11, Val=3, Test=3


# 4. Архитектуры моделей

In [ ]:
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

 (1): Linear(in_features=1280, out_features=1000, bias=True) - на выходе 1000 слоев

In [ ]:
for param in model.parameters():
#    print(param.)
    param.requires_grad = False

In [ ]:
#сохораняем вол-входов на входе  = пред. выход 
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 10, bias=True) #- меняем на 10

print(model.classifier[1])

Linear(in_features=1280, out_features=10, bias=True)


In [ ]:
# Проверка: теперь градиенты будут считаться только для нового слоя
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Обучаемый слой: {name}")

Обучаемый слой: classifier.1.weight
Обучаемый слой: classifier.1.bias


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Используемое устройство: {device}')

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3) #, weight_decay=1e-2)

epochs = 30  # Обучение: 30 эпох

Используемое устройство: cpu


# 5. Дообучение 

In [ ]:
gl_train_losses = []
gl_train_accuracies = []
gl_val_losses = []
gl_val_accuracies = []
    
for epoch in range(epochs):
    # --- Фаза обучения ---
    model.train()
    
    train_loss = 0.0
    correct_train = 0
    total_train = 0



    # идем циклом по картинкам для обучения 
    for images, labels in train_loader_images:
        # переносим данные на устройсво (которе выбрали выше что смогли скорее)
        images, labels = images.to(device), labels.to(device)

        #обнуляем градиенты 
        optimizer.zero_grad()

        # проходим прямой проход 
        outputs = model(images)
        # Считаем потери criterion - функция потерь
        loss = criterion(outputs, labels) 
        # проходим обратный проход 
        loss.backward()

        # обновляем веса в моделе
        optimizer.step()

        # расчет ошибки
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        #сравниванем предсказание с эталоном - (labels)
        correct_train += predicted.eq(labels).sum().item()

    

    # считаем общую статистику
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = 100.0 * correct_train / total_train

    gl_train_losses.append(epoch_train_loss)
    gl_train_accuracies.append(epoch_train_acc)

    # --- валидация  ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    # отключаем градиент - ибо валидация
    with torch.no_grad():
        for images, labels in val_loader_images:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = 100.0 * correct_val / total_val

    gl_val_losses.append(epoch_val_loss)
    gl_val_accuracies.append(epoch_val_acc)
    
    # 
    if (epoch + 1) == 1 or (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
        print(f"Эпоха [{epoch+1}/{epochs}]")
        print(f"  Обучение -> Потери: {epoch_train_loss:.4f} | Точность: {epoch_train_acc:.2f}%")
        print(f"  Валидация -> Потери: {epoch_val_loss:.4f} | Точность: {epoch_val_acc:.2f}%")
        print("-" * 50)

Эпоха [1/30]
  Обучение -> Потери: 2.1905 | Точность: 22.75%
  Валидация -> Потери: 2.1862 | Точность: 26.17%
--------------------------------------------------
Эпоха [5/30]
  Обучение -> Потери: 1.3959 | Точность: 64.09%
  Валидация -> Потери: 1.6478 | Точность: 45.64%
--------------------------------------------------
Эпоха [10/30]
  Обучение -> Потери: 1.1788 | Точность: 72.82%
  Валидация -> Потери: 1.5182 | Точность: 51.01%
--------------------------------------------------


libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x111cbc220>
Traceback (most recent call last):
  File "/Users/grushinda/miniconda3/envs/nn_project/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/Users/grushinda/miniconda3/envs/nn_project/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1568, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/Users/grushinda/miniconda3/envs/nn_project/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/grushinda/miniconda3/envs/nn_project/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.senti

KeyboardInterrupt: 

# 6. Тестирование модели

In [ ]:
model.eval()
test_loss = 0.0
correct_test = 0
total_test = 0

gl_labels = []
gl_predict = []
    
with torch.no_grad():
    for images, labels in test_loader_images:
        images, labels = images.to(device), labels.to(device)
            
        outputs = model(images)
        loss = criterion(outputs, labels)
            
        test_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()
        
        gl_predict.extend(predicted.cpu().numpy())
        gl_labels.extend(labels.cpu().numpy())
            
total_test_loss = test_loss / total_test
total_test_acc = 100.0 * correct_test / total_test
    
print(f"  Тест     -> Потери: {total_test_loss:.4f} | Точность: {total_test_acc:.2f}%")
print("-" * 50)

  Тест     -> Потери: 1.4829 | Точность: 54.97%
--------------------------------------------------
